In [42]:
# Santander Customer Satisfaction - Decision Tree Classification

In [43]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

In [44]:
# Load datasets
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

In [45]:
# Separate features and target
X_train = train_df.drop(columns=["TARGET"], errors="ignore")
y_train = train_df.get("TARGET")

X_test = test_df.drop(columns=["TARGET"], errors="ignore")
y_test = test_df.get("TARGET")

In [46]:
# Keep only rows with valid target values
if y_train is not None:
    train_valid_idx = y_train.notna()
    X_train_clean = X_train.loc[train_valid_idx]
    y_train_clean = y_train.loc[train_valid_idx]
else:
    X_train_clean = X_train
    y_train_clean = None

if y_test is not None:
    test_valid_idx = y_test.notna()
    X_test_clean = X_test.loc[test_valid_idx]
    y_test_clean = y_test.loc[test_valid_idx]
else:
    X_test_clean = X_test
    y_test_clean = None

In [47]:
# Train Decision Tree Classifier
clf = DecisionTreeClassifier(splitter="random", random_state=42)

if y_train_clean is not None and not y_train_clean.empty:
    clf.fit(X_train_clean, y_train_clean)

In [49]:
# %% Predictions and probabilities
def get_predictions_df(clf, X):
    pred = clf.predict(X)
    prob = clf.predict_proba(X)
    df = pd.concat([
        pd.DataFrame(pred, columns=["Prediction"]),
        pd.DataFrame(prob, columns=[f"Prob_Class_{i}" for i in range(prob.shape[1])])
    ], axis=1)
    return df, pred

train_results_df = pd.DataFrame()
train_pred = None
test_results_df = pd.DataFrame()
test_pred = None

# Only predict if classifier is fitted
if y_train_clean is not None and not y_train_clean.empty:
    train_results_df, train_pred = get_predictions_df(clf, X_train_clean)
    
if y_test_clean is not None and not y_test_clean.empty:
    test_results_df, test_pred = get_predictions_df(clf, X_test_clean)

In [50]:
# Classification reports
train_report_df = pd.DataFrame()
test_report_df = pd.DataFrame()

if y_train_clean is not None:
    train_report = classification_report(y_train_clean, train_pred, output_dict=True)
    train_report_df = pd.DataFrame(train_report).transpose()

if y_test_clean is not None:
    test_report = classification_report(y_test_clean, test_pred, output_dict=True)
    test_report_df = pd.DataFrame(test_report).transpose()

In [51]:
# Save all outputs to Excel
output_file = "data/Output_Santander.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    train_results_df.to_excel(writer, sheet_name="Train_Predictions", index=False)
    test_results_df.to_excel(writer, sheet_name="Test_Predictions", index=False)
    if not train_report_df.empty:
        train_report_df.to_excel(writer, sheet_name="Train_Report")
    if not test_report_df.empty:
        test_report_df.to_excel(writer, sheet_name="Test_Report")

print(f"All outputs saved to {output_file}")

All outputs saved to data/Output_Santander.xlsx
